## **Install Required Libraries**

Installed all required libraries:

sentence-transformers → for SBERT embeddings
rank-bm25 → for keyword-based retrieval
google-generativeai → for Gemini AP

In [ ]:
!pip install sentence-transformers rank-bm25 google-generativeai

## **Import Libraries**

Imported all necessary modules:

NumPy → numerical operations
BM25 → keyword search
SBERT → semantic search
CrossEncoder → re-ranking
Gemini → LLM generation

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import google.generativeai as genai

## **CELL 3 :Setup Gemini API**

Configured Gemini API using API key

Loaded model: gemini-2.5-flash

This model is used for:

Query expansion

Final answer generation

In [ ]:
genai.configure(api_key="enter_gemini_api_key")

gemini_model = genai.GenerativeModel("models/gemini-2.5-flash")

print("Gemini Loaded Successfully ")

## **CELL 4: Create Document Corpus**

Created a dataset of ~20 AI/ML documents

Ensured:

Multiple documents per topic

Included technical terms (AdamW, BERT, etc.)

This corpus is used for retrieval

In [ ]:
corpus = [
    "Transformers use self-attention mechanisms to understand relationships between words in a sentence.",
    "Self-attention assigns different importance to words when encoding meaning.",
    "Multi-head attention allows transformers to focus on different parts of a sentence simultaneously.",
    "Positional encoding helps transformers understand the order of words in a sequence.",

    "Gradient descent is an optimization algorithm used to minimize loss in neural networks.",
    "Adam optimizer improves gradient descent by adapting learning rates.",
    "Stochastic gradient descent updates weights using small batches of data.",
    "Learning rate controls how fast a model learns during training.",

    "Backpropagation is used to update weights in neural networks using error gradients.",
    "Loss functions measure the difference between predicted and actual outputs.",
    "Batch normalization stabilizes training and improves convergence speed.",

    "Overfitting occurs when a model learns training data too well and fails on new data.",
    "Regularization techniques like dropout help prevent overfitting.",
    "L2 regularization adds a penalty to large weights to improve generalization.",

    "BERT is a transformer-based model designed for understanding context in language.",
    "GPT models are autoregressive and generate text based on previous tokens.",

    "Tokenization converts text into smaller units for processing.",
    "Stopwords are common words that are often removed during text preprocessing.",

    "AdamW optimizer is a variant of Adam used in transformer training.",
    "Cross-entropy loss is commonly used for classification tasks."
]

print("Total Documents:", len(corpus))
print("Sample Doc:", corpus[0])

## **CELL 5: Load Embedding & Re-Ranking Models**

Loaded:

SBERT (all-MiniLM-L6-v2) → for semantic search

Cross Encoder → for re-ranking results

In [ ]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Models Loaded ")

## **CELL 6: Test SBERT Embedding**

Converted a query into embedding vector

Verified output shape (384,)

Ensured SBERT is working correctly

In [ ]:
query = "what is attention"
embedding = sbert_model.encode(query)

print("Embedding shape:", embedding.shape)

## **CELL 7: Setup BM25**

Tokenized corpus (lowercase + split)

Initialized BM25 model

Enables keyword-based retrieval

In [ ]:
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 Ready ")

## **CELL 8: Test BM25 Retrieval**

Tested BM25 with a sample query

Checked top matching document

Verified keyword search works

In [ ]:
query = "attention mechanism"
tokenized_query = query.lower().split()

scores = bm25.get_scores(tokenized_query)
top_doc = corpus[np.argmax(scores)]

print("BM25 Top Doc:", top_doc)

## **CELL 9: Implement Hybrid Retriever**


Combined:

BM25 (keyword search)

SBERT (semantic search)

Used RRF (Reciprocal Rank Fusion) to merge results

Returned:

doc_id
rrf_score
bm25_rank
sbert_rank
text

In [ ]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        self.embeddings = sbert_model.encode(corpus, convert_to_numpy=True)

    def retrieve(self, query, top_k=5):

        # BM25
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        # SBERT
        query_embedding = sbert_model.encode(query, convert_to_numpy=True)
        sbert_scores = np.dot(self.embeddings, query_embedding)
        sbert_ranks = np.argsort(sbert_scores)[::-1]

        # ✅ Weighted RRF
        scores = {}

        dense_weight = 0.6
        sparse_weight = 0.4

        for rank, doc_id in enumerate(bm25_ranks):
            scores[doc_id] = scores.get(doc_id, 0) + sparse_weight * (1 / (self.k + rank + 1))

        for rank, doc_id in enumerate(sbert_ranks):
            scores[doc_id] = scores.get(doc_id, 0) + dense_weight * (1 / (self.k + rank + 1))

        ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for doc_id, score in ranked_docs[:top_k]:
            results.append({
                "doc_id": doc_id,
                "rrf_score": score,
                "bm25_rank": int(np.where(bm25_ranks == doc_id)[0][0] + 1),
                "sbert_rank": int(np.where(sbert_ranks == doc_id)[0][0] + 1),
                "text": self.corpus[doc_id]
            })

        return results

## **CELL 10: Test Hybrid Retrieval**

Ran hybrid retrieval on query

Verified:

ranks are present

correct document retrieved

Ensured fusion works properly

In [ ]:
retriever = HybridRetriever(corpus)

results = retriever.retrieve("attention mechanism")

for r in results:
    print(r)

## **CELL 11: Implement Cross-Encoder Re-Ranker**

Compared query + document pairs
Assigned relevance scores

Sorted documents based on score
Selected top-k results

In [ ]:
def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]

    scores = cross_encoder.predict(pairs)

    for i, doc in enumerate(candidates):
        doc["cross_score"] = scores[i]

    sorted_docs = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)

    return sorted_docs[:top_k]

## **CELL 12: Test Re-Ranking**

Applied re-ranking on retrieved results

Verified:
scores (even negative)

correct sorting

Ensured best document comes first

In [ ]:
reranked = rerank("attention mechanism", results)

for r in reranked:
    print(r["cross_score"], "->", r["text"])

## **CELL 13: Query Expansion using Gemini**

Used Gemini to generate 3 rephrased queries

Helps handle vague queries

Improves retrieval performance

In [ ]:
def generate_queries(query):
    prompt = f"""
    Generate exactly 3 different rephrased versions of the following query.
    Return each query on a new line.

    Query: {query}
    """

    response = gemini_model.generate_content(prompt)

    text = response.text.strip()

    queries = [q.strip("- ").strip() for q in text.split("\n") if q.strip()]

    return queries[:3]

## **CELL 14: Test Query Expansion**

Generated multiple query variations

Verified output format

Ensured clean and usable queries

In [ ]:
queries = generate_queries("what is attention")

print("Generated Queries:")
for q in queries:
    print(q)

## **CELL 15: Build Advanced RAG Pipeline**

Combined all components:

Query Expansion

Hybrid Retrieval

Re-ranking

Gemini Answer Generation

Created full end-to-end pipeline

In [ ]:
def advanced_rag(user_query):

    # Step 1: Query Expansion
    expanded_queries = generate_queries(user_query)

    all_results = []

    # Step 2: Retrieval
    for q in expanded_queries:
        res = retriever.retrieve(q, top_k=5)
        all_results.extend(res)

    # Step 3: Remove duplicates
    unique_docs = {doc["text"]: doc for doc in all_results}.values()

    # Step 4: Re-ranking
    reranked = rerank(user_query, list(unique_docs))

    # Step 5: LLM Generation (Gemini)
    context = "\n".join([doc["text"] for doc in reranked])

    prompt = f"""
    Answer the question based on the context below:

    Context:
    {context}

    Question:
    {user_query}
    """

    response = gemini_model.generate_content(prompt)

    return response.text, reranked

## **CELL 16: Test Advanced RAG**

Ran full pipeline on a query

Generated:
final answer

top documents

Verified complete system works

In [ ]:
answer, docs = advanced_rag("how does attention work")

print("ANSWER:\n", answer)

print("\nTop Docs:")
for d in docs:
    print(d["text"])

## **CELL 17: Implement Naïve RAG**

Used only SBERT for retrieval

No expansion, no re-ranking

Serves as baseline for comparison

In [ ]:
def naive_rag(query):
    query_embedding = sbert_model.encode(query, convert_to_numpy=True)

    corpus_embeddings = sbert_model.encode(corpus, convert_to_numpy=True)

    scores = np.dot(corpus_embeddings, query_embedding)

    top_doc_id = np.argmax(scores)

    return corpus[top_doc_id]

## **CELL 18: Compare Naïve vs Advanced RAG**

Ran both pipelines on same queries

Compared top documents

Checked whether results differ

In [ ]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is overfitting in machine learning?"
]

results = []

for q in queries:
    naive_doc = naive_rag(q)

    adv_answer, adv_docs = advanced_rag(q)
    adv_doc = adv_docs[0]["text"]

    results.append((q, naive_doc, adv_doc, naive_doc != adv_doc))

    print("\n==============================")
    print("Query:", q)
    print("Naive:", naive_doc)
    print("Advanced:", adv_doc)
    print("Different:", naive_doc != adv_doc)

## **CELL 19: Display Results Table**

Converted results into DataFrame

Displayed comparison table clearly

Helps analyze improvement

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=[
    "Query",
    "Naive RAG Top Doc",
    "Advanced RAG Top Doc",
    "Are they different?"
])

df

| Query | Naive RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|------|------------------|---------------------|---------------------|
| optimization techniques for training | Batch normalization stabilizes training and improves convergence speed. | Gradient descent is an optimization algorithm used to minimize loss in neural networks. | Yes |
| what is overfitting in machine learning? | Overfitting occurs when a model learns training data too well and fails on new data. | Overfitting occurs when a model learns training data too well and fails on new data. | No |
| how do transformers encode meaning? | Positional encoding helps transformers understand the order of words in a sequence. | Positional encoding helps transformers understand the order of words in a sequence. | No |